# Save Your Work

Before starting, save this notebook to your Google Drive:
1. Click **File** → **Save a copy in Drive**
2. The copy will open automatically
3. Work in the Google Drive copy from now on

---

# SQL and Python

Integrate SQL queries into Python data science workflows using sqlite3 and Pandas.

In [ ]:
import sqlite3
import pandas as pd

## Basic workflow: connect, query, fetch

In [ ]:
# Connect to bookstore database
with sqlite3.connect("bookstore.db") as conn:
    cursor = conn.cursor()
    
    # Execute query
    cursor.execute("SELECT name, city FROM customers")
    
    # Fetch all results
    results = cursor.fetchall()
    print(f"Found {len(results)} customers:")
    for row in results:
        print(f"  {row[0]} - {row[1]}")

## Parameterized queries (safe from SQL injection)

In [ ]:
with sqlite3.connect("bookstore.db") as conn:
    cursor = conn.cursor()
    
    # Use ? placeholder for safe parameter binding
    city = 'Chicago'
    cursor.execute("SELECT name, email FROM customers WHERE city = ?", (city,))
    
    print(f"Customers in {city}:")
    for row in cursor.fetchall():
        print(f"  {row[0]} ({row[1]})")

## Load into Pandas DataFrame

In [ ]:
with sqlite3.connect("bookstore.db") as conn:
    # Direct SQL query to Pandas
    df = pd.read_sql_query("SELECT * FROM books", conn)
    
    print("Books DataFrame:")
    print(df)
    print(f"\nShape: {df.shape}")
    print(f"Columns: {list(df.columns)}")

## Complex query with joins to Pandas

In [ ]:
with sqlite3.connect("bookstore.db") as conn:
    query = """
    SELECT
        c.name as customer,
        b.title as book,
        b.price,
        o.quantity,
        (b.price * o.quantity) as total
    FROM customers c
    JOIN orders o ON c.customer_id = o.customer_id
    JOIN books b ON o.book_id = b.book_id
    ORDER BY o.order_date
    """
    
    df_orders = pd.read_sql_query(query, conn)
    print("Order details in Pandas:")
    print(df_orders)
    
    # Now you can analyze in Pandas
    print(f"\nTotal revenue: ${df_orders['total'].sum():.2f}")
    print(f"\nRevenue by customer:")
    print(df_orders.groupby('customer')['total'].sum())

## Insert data from Python

In [ ]:
with sqlite3.connect("bookstore.db") as conn:
    cursor = conn.cursor()
    
    # Insert a new customer
    new_customer = ('Taylor Kim', 'taylor@email.com', 'Denver', '2024-01-20')
    cursor.execute(
        "INSERT INTO customers (name, email, city, join_date) VALUES (?, ?, ?, ?)",
        new_customer
    )
    
    conn.commit()
    print(f"Inserted new customer: {new_customer[0]}")
    
    # Verify
    cursor.execute("SELECT COUNT(*) FROM customers")
    count = cursor.fetchone()[0]
    print(f"Total customers now: {count}")